<05_More_Crawling.ipynb>

제미나이 의존도: 40-50%

전에 만들었던 신선한 사과 분류기 모델에서 데이터가 너무 적었기 때문에, 이번에는 많은 데이터를 가져와서 학습하고 싶었습니다.

지난번 크롤링에서 리캡차를 피할 수 없었기 때문에 리캡차를 피할 수 있는 다른 방법을 찾았습니다. undetected_chromedriver(uc)
uc는 웹사이트의 봇 감지 시스템을 우회하기 위해 크롬드라이버를 변조한 라이브러리입니다.
기존 셀레니움은 브라우저 실행시 "자동화 도구에 의해 제어되고 있습니다"라는 흔적이 남았기 때문에 리캡차를 피할 수 없었지만, 
ucs는 이러한 자동화 흔적을 지우고 일반 사용자가 브라우저를 켠 것처럼 위장해 줍니다.

더 나아가서, 
1. 사이트 접속
2. 검색어 입력 및 엔터
3. 이미지 탭 이동

위 3가지 과정을 한번에 통합하는 방식을 고안해서..
URL 쿼리 스트링 파라미터 제어를 통해 단번에 이미지 탭에 들어오게 됩니다.

구글의 이미지 탭에서 스크롤을 내리게 해서 HTML 문서에서 더 많은 이미지를 긁어오게 합니다.

이미지 태그(img) 중에는 학습에 불필요하거나 좋지 않은 데이터도 있기 때문에, 가로 및 세로가 128 이상인 녀석들로 엄선했습니다.

이렇게 861(447 + 414)개의 데이터가 모였습니다.

In [4]:
import undetected_chromedriver as uc
import time
import urllib.parse

options = uc.ChromeOptions()
driver = uc.Chrome(options=options, version_main=149)
driver.set_window_size(768, 768)

search_query = urllib.parse.quote("신선한 사과")
driver.get(f"https://google.com/search?q={search_query}&tbm=isch")
time.sleep(2)

In [6]:
# 스크롤 5번 -> 조금 대기(sleep) -> 이미지 크롤링
for i in range(5):
    # 브라우저의 Y축 스크롤 위치를 현재 페이지의 총 높이(scrollHeight)로 이동시켜라
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
    # 스크롤을 내린 후 새 이미지가 뜨는 데 걸리는 '로딩 시간'을 줘야 합니다.
    time.sleep(2.0)

0
1
2
3
4


In [10]:
import re
from bs4 import BeautifulSoup

# 현재 브라우저의 HTML 소스를 가져와 뷰티불수프에게 넘긴다.
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

target_images = soup.find_all('img', src=re.compile(r'^https://encrypted'))

print(len(target_images))

850


In [15]:
import requests
from PIL import Image
from io import BytesIO
import random

img_src_list = []

for i, image in enumerate(target_images):
    src = image['src']
    try:
        # requests를 이용해 이미지의 raw 데이터를 인터넷에서 잠시 읽어온다.
        response = requests.get(src, timeout=5)

        # 읽어온 바이너리 데이터를 PIL이 읽을 수 있는 이미지 객체로 변환한다.
        img = Image.open(BytesIO(response.content))

        # PIL 이미지 객체는 .size 속성을 통해 (가로, 세로) 튜플을 뱉어낸다.
        width, height = img.size

        if width >= 128 and height >= 128:
            img.save(f"more_data/fresh_apple/fresh_{i}.jpg")
            time.sleep(random.uniform(0.5, 1.5))
    except Exception as e:
        continue

# print(len(img_src_list))

In [17]:
import os
path = 'more_data/fresh_apple/'
dir_list = os.listdir(path)
print(len(dir_list))

447


In [27]:
search_query = urllib.parse.quote("싱싱하지 않은 사과")
driver.get(f"https://google.com/search?q={search_query}&tbm=isch")
time.sleep(2)

In [30]:
for i in range(5):
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight)')
    time.sleep(2.0)

In [31]:
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

target_images = soup.find_all('img', src=re.compile(r'https://encrypted'))
print(len(target_images))

873


In [ ]:
for i, image in enumerate(target_images):
    src = image['src']
    try:
        response = requests.get(src, timeout=5)
        img = Image.open(BytesIO(response.content))

        width, height = img.size

        if width >= 128 and height >= 128:
            img.save(f"more_data/rotten_apple/rotten_{i}.jpg")
            time.sleep(random.uniform(0.5, 1.5))
        else:
            print(f"{i} - width: {width}, height: {height}")
    except Exception as e:
        print(f"{i} - {e}")
        continue

print("크롤링 완료")

0 - width: 69, height: 46
13 - cannot write mode P as JPEG
18 - width: 50, height: 50
19 - width: 50, height: 50
20 - width: 50, height: 50
26 - width: 16, height: 16
28 - width: 16, height: 16
30 - width: 15, height: 16
32 - width: 16, height: 16
34 - width: 16, height: 16
36 - width: 16, height: 16
38 - width: 16, height: 16
40 - width: 16, height: 16
42 - width: 16, height: 16
44 - width: 16, height: 16
46 - width: 16, height: 16
48 - width: 16, height: 16
50 - width: 16, height: 16
52 - width: 16, height: 16
54 - width: 16, height: 16
56 - width: 16, height: 16
58 - width: 16, height: 16
60 - width: 16, height: 16
62 - width: 16, height: 16
64 - width: 16, height: 16
65 - width: 50, height: 50
66 - width: 50, height: 50
67 - width: 50, height: 50
69 - width: 16, height: 16
71 - width: 16, height: 16
73 - width: 16, height: 16
75 - width: 16, height: 16
77 - width: 16, height: 16
79 - width: 16, height: 16
81 - width: 16, height: 16
83 - width: 16, height: 16
85 - width: 16, height:

In [2]:
import os
dir_list = os.listdir('more_data/rotten_apple/')
print(len(dir_list))

414
